In [36]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np

ol = Overlay("rhd_spi_ip_0806_2_wrapper.bit")

In [37]:
ol?

In [45]:
# 名稱以你的 block design 為準；在 Vivado Address Editor 可以看到
spi_ip = ol.axi_rhd_spi_master_0   # 假設 instance 名稱為 “…_0”

# 也可列出所有暫存器 (32-bit words)
spi_ip.register_map

AttributeError: register_map only available if the .hwh is provided

In [23]:
# RHD SPI IP 的基地址 
RHD_SPI_BASE = 0xA0000000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
PHA_SEL_REG = 0x04  # [3:0]
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data
STATUS_REG  = 0x10  # bit[0]=busy, bit[1]=finish
# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x00000001
Busy: 1
Finish: 0


In [24]:
mmio.write(CONTROL_REG, 0x01)

In [25]:
mmio.write(PHA_SEL_REG, 0x02)

In [26]:
mmio.read(STATUS_REG)

1

In [38]:
import time

# 假設 mmio 是一個已經初始化的 MMIO 實例，並且 RX_DATA_REG 是接收資料的寄存器地址
data_list = []

# 讀取 100 筆資料，每筆資料 16 位元
for _ in range(10):
    data = mmio.read(TX_DATA_REG)  # 讀取 16 位元的資料
    data_list.append(data)  # 將資料加入到列表中

# 現在 data_list 包含了 100 筆接收到的 16-bit 資料
print(data_list)


[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [40]:
# 2) 连续抓 10 筆
samples = []
while len(samples) < 10:
    data = spi_ip.RX_DATA & 0xFFFF
    samples.append(data)
        # 等 FINISH 归 0 → 避免重複记录同一筆
                   # 或 time.sleep(1e-6)
print("10 samples:", samples)


AttributeError: 'DefaultIP' object has no attribute 'RX_DATA'

In [41]:
# 假設 register_map 已能使用
dir(spi_ip.register_map)
# 或
spi_ip.register_map


AttributeError: register_map only available if the .hwh is provided

In [44]:
ol = Overlay("rhd_spi_ip_0806_wrapper.bit")
# 成功載入 HWH 時，會多出 description/ip_dict
print("Has description:", hasattr(ol, "description"))
print("IP list =====")
for n, v in ol.ip_dict.items():
    print(f"{n:50s} @ 0x{v['phys_addr']:08X}")

Has description: False
IP list =====
axi_rhd_spi_master_0                               @ 0xA0000000


KeyError: 'phys_addr'

In [43]:
from pynq import MMIO
BASE = 0x40000000       # ← 依你的 design 修改
spi  = MMIO(BASE, 0x1000)

RX_DATA = 0x0C
STATUS  = 0x10

samples = []
while len(samples) < 10:
    if spi.read(STATUS) & 0x2:          # FINISH = 1
        samples.append(spi.read(RX_DATA) & 0xFFFF)
        # 等 FINISH 歸 0
        while spi.read(STATUS) & 0x2:
            pass
print(samples)


PermissionError: [Errno 1] Operation not permitted